# Flujo de trabajo en Python

**Unidad 4.a y 4.b del temario** · Se hace **una vez, al inicio del semestre**

Antes de la primera actividad hay que dejar el ambiente en un estado conocido. No es
un trámite: buena parte de los problemas que aparecen en un curso de este tipo son
diferencias de ambiente que nadie registró.

**La actividad no consiste en que todo salga en verde, sino en saber leer lo que
reporta.** Un cuaderno que corre en una máquina y falla en otra casi siempre difiere
en algo de lo que este cuaderno imprime.

In [1]:
import importlib
import sys
from pathlib import Path

# Raíz del repositorio: dos niveles arriba de esta carpeta.
RAIZ = Path.cwd().parents[1]

print(f"Python {sys.version.split()[0]}")
print(f"Intérprete: {sys.executable}")
print(f"Repositorio: {RAIZ}")

Python 3.9.6
Intérprete: /Library/Developer/CommandLineTools/usr/bin/python3
Repositorio: /Users/benjamin/Documents/Personal/Cursos_UNAM/Econometria_I_2026/Econometria-I-2026


## 1. Qué intérprete estás usando

La ruta de arriba importa. En macOS es normal tener varios Python instalados —el del
sistema, el de Homebrew, el de Anaconda— y el cuaderno usa **uno** de ellos. Cuando una
biblioteca «está instalada» pero el cuaderno no la encuentra, casi siempre es que se
instaló en otro intérprete.

Si necesitas instalar algo desde el cuaderno, la forma robusta es:

```python
import sys
!{sys.executable} -m pip install nombre_del_paquete
```

Así se instala en el intérprete del kernel activo y no en otro.

## 2. Versiones de las bibliotecas

Anótalas. Cuando un resultado no se reproduzca, es el primer lugar donde hay que mirar.

In [2]:
REQUERIDAS = {
    "numpy": "1.24",
    "pandas": "2.0",
    "statsmodels": "0.14",
    "scipy": "1.10",
    "matplotlib": "3.7",
}

OPCIONALES = {
    "seaborn": "visualización en varios cuadernos",
    "linearmodels": "datos panel (Clase_04) y sistemas de ecuaciones (Clase_03)",
    "sklearn": "aprendizaje estadístico (Clase_14, material adicional)",
}

# El nombre con que se importa no siempre es el nombre con que se instala.
NOMBRE_PIP = {"sklearn": "scikit-learn"}


def como_tupla(version):
    """'2.2.3' -> (2, 2, 3). Ignora sufijos como 'rc1' o 'dev0'."""
    partes = []
    for pieza in version.split("."):
        digitos = ""
        for caracter in pieza:
            if caracter.isdigit():
                digitos += caracter
            else:
                break
        if digitos == "":
            break
        partes.append(int(digitos))
    return tuple(partes)


def revisar(nombre, minima=None, nota=None):
    """Importa el módulo y reporta su versión. Devuelve True si cumple."""
    try:
        modulo = importlib.import_module(nombre)
    except ImportError:
        detalle = f"  ({nota})" if nota else ""
        paquete = NOMBRE_PIP.get(nombre, nombre)
        print(f"  FALTA    {nombre:14s} instalar con: pip install {paquete}{detalle}")
        return False

    version = getattr(modulo, "__version__", "desconocida")
    if minima is None:
        print(f"  presente {nombre:14s} {version}")
        return True
    if version == "desconocida" or como_tupla(version) >= como_tupla(minima):
        print(f"  ok       {nombre:14s} {version}")
        return True
    print(f"  ANTIGUA  {nombre:14s} {version} (se probó con {minima} o superior)")
    return False


print("Bibliotecas necesarias")
faltantes = [n for n, v in REQUERIDAS.items() if not revisar(n, minima=v)]

print("\nBibliotecas opcionales")
for nombre, nota in OPCIONALES.items():
    revisar(nombre, nota=nota)

Bibliotecas necesarias
  ok       numpy          1.26.4
  ok       pandas         2.2.3
  ok       statsmodels    0.14.6
  ok       scipy          1.13.1
  ok       matplotlib     3.9.4

Bibliotecas opcionales
  presente seaborn        0.13.2
  FALTA    linearmodels   instalar con: pip install linearmodels  (datos panel (Clase_04) y sistemas de ecuaciones (Clase_03))
  presente sklearn        1.6.1


## 3. Los datos del curso están donde deben

Los cuadernos usan **rutas relativas** (`../../Clase_04_DatosPanel/wage_panel.csv`), no
rutas absolutas con tu nombre de usuario. Es lo que permite que corran en otra máquina.

Para que funcionen hay que **clonar el repositorio completo**, no descargar archivos
sueltos desde la interfaz web de GitHub.

In [3]:
DATOS = [
    "Clase_01_RegresionLineal/nerlove63.dta",
    "Clase_02_VariablesInstrumentales/maketable4.dta",
    "Clase_04_DatosPanel/wage_panel.csv",
    "Clase_05_EleccionBinaria/Mroz.csv",
    "Clase_11_DiferenciaEnDiferencias/employment.csv",
    "Clase_12_ControlSintetico/smoking_data.csv",
]

sin_datos = []
for relativa in DATOS:
    ruta = RAIZ / relativa
    if ruta.exists():
        print(f"  ok    {relativa}  ({ruta.stat().st_size / 1024:.0f} KB)")
    else:
        print(f"  FALTA {relativa}")
        sin_datos.append(relativa)

print()
if not faltantes and not sin_datos:
    print("Ambiente completo. Se puede continuar con 01_Escribir_con_especificacion.")
else:
    if faltantes:
        print("Instalar lo que falta:  pip install " + " ".join(faltantes))
    if sin_datos:
        print("Faltan datos: clonar el repositorio completo.")

  ok    Clase_01_RegresionLineal/nerlove63.dta  (4 KB)
  ok    Clase_02_VariablesInstrumentales/maketable4.dta  (12 KB)
  ok    Clase_04_DatosPanel/wage_panel.csv  (183 KB)
  ok    Clase_05_EleccionBinaria/Mroz.csv  (27 KB)
  ok    Clase_11_DiferenciaEnDiferencias/employment.csv  (5 KB)
  ok    Clase_12_ControlSintetico/smoking_data.csv  (57 KB)

Ambiente completo. Se puede continuar con 01_Escribir_con_especificacion.


## 4. Buenas prácticas que se dan por establecidas desde aquí

- **Rutas relativas, siempre.**
- **Clonar el repositorio completo**, no bajar archivos sueltos.
- **Un ambiente declarado:** `requirements.txt` está en la raíz de esta carpeta. Para el
  proyecto final, la plantilla fija las versiones con `==` y no con `>=`: es la
  diferencia entre «corre» y «da el mismo número».
- **Reiniciar el kernel y ejecutar todo** antes de dar un cuaderno por terminado
  (*Kernel → Restart & Run All*). Un cuaderno que depende del orden en que se
  ejecutaron las celdas no es reproducible, aunque en pantalla se vea bien.

---
Parte del curso de **Econometría I**, Facultad de Ciencias, UNAM.
Ver el [README de la carpeta](README.md) para el calendario de actividades.